## Print information about a tuned chorale
1. Chorale name, tolerance, tonal diamond shape, limit max
1. Cent values, note names, scores, and ratios for every chord
2. Top notes cents, note names, cent values


In [1]:
import os
local_dir = os.getcwd()  # Current working directory
print(f'The local directory is: {local_dir}')

The local directory is: /home/prent/Repos/One-footed-bride-tuning


In [2]:
import logging, os, sys, time
from importlib import reload
import numpy as np
from importlib import reload
from collections import Counter, defaultdict
user = 'prent'
local_dir = os.getcwd()  # Current working directory
print(f'The local directory is: {local_dir}')
base_dir = local_dir
WAVE_DIR = os.path.join(base_dir, 'Music', 'sflib')
# The latest files are here: Archive/straw-man/t1_r1.75_s2.50_md28_sn10/bwv253-opt.npy
numpy_dir = os.path.join(base_dir, 'Archive', 'straw-man')

np.set_printoptions(legacy='1.25')
import diamond_music_utils as dmu
import adaptive_tuning_util as atu 
from itertools import count, combinations, permutations
dmu.start_logger('test.log',log_level = 'info') # how to modify this so that it only prints to the log and not in the notebook.
logging.info(f'{base_dir = }, {numpy_dir = }, {WAVE_DIR = }')
rng = np.random.default_rng()

The local directory is: /home/prent/Repos/One-footed-bride-tuning


In [3]:
print(f'{np.power(2, (1/12)):.6f}')

1.059463


In [4]:
def print_chords(version, input_file, numpy_dir, measure, tolerance, ratios=True, print_individual_chords=True,\
            offset=0, use_werck_top_notes=False, print_top_notes = True):
    
    _, _, chorale, root, mode, keys = atu.load_chorale_in_cents(version, numpy_dir)
    try:
        floating_cents = np.load(input_file)
        existing_chorale_in_cents = np.rint(floating_cents).astype(int)
    except:
        print(f'Trouble loading {input_file = }')
        return {input_file}
    if use_werck_top_notes:
        input_file = os.path.join(numpy_dir, f'{version}-w-top_notes.npy')
    else: 
        input_file = os.path.join(numpy_dir, f'{version}top-notes.npy')
        if print_top_notes:
            print(f'Loaded top_notes from {input_file = }')
    try:      
        top_notes = np.load(input_file)
    except:
        print(f'Trouble loading {input_file = }')
        return {input_file}
    top_notes[1] = top_notes[1] + offset
    
    if print_top_notes:
        print(f'Key: {keys[root]} {mode}, {tolerance = }')
        print(f'\ntop notes:')
        print(*[inx for inx in np.arange(12)], sep='\t')
        print(*[note for note in top_notes[0]], sep = '\t')
        print(*[keys[note] for note in top_notes[0]], sep = '\t')
        print(*[cent_value for cent_value in top_notes[1]], sep = '\t')
    if print_individual_chords: 
        print(f'\n#          cents       note names   chord score')
        # #     +----- cents -----+--- note names---+--- chord score'
        # 0:    0  386    0  884\tC♮ E♮ C♮ A♮\t47.0
    if measure > 0: print(f'\nprinting only measure {measure}')
    prev_chord = np.zeros(4, dtype=int)
    header1 = f" # Fr/To Cents Ratio\t # Fr/To Cents Ratio\t # Fr/To Cents Ratio"
    
    for inx, chord_in_cents in zip(count(0,1), existing_chorale_in_cents.T):
        if not np.array_equal(prev_chord, chord_in_cents):
            if measure == 0 or 16 * (measure - 1) <= inx < 16 * measure:
                tuned_pcs = np.array(atu.pitch_class_from_cents(chord_in_cents), dtype=int) % 12
                if print_individual_chords: 
                        # Print tuned note names (from cents), not original MIDI pitch classes.
                        pitches = ' '.join(map(str, keys[tuned_pcs]))
                        print(f'{inx}: {atu.format_chord(chord_in_cents,4)}\t{pitches}\t{chord_scorer.score_chord(chord_in_cents, tolerance=tolerance)}')
                if ratios:
                    print(f'{header1}')
                    intervals = []
                    for inx1, inx2 in combinations(np.arange(4),2):
                            cent_value_interval_pair = np.array([chord_in_cents[inx1], chord_in_cents[inx2]])
                            cent_value_delta, cent_value_moves, cent_value_target = atu.cent_value_interval(cent_value_interval_pair)
                            best_idx = chord_scorer.find_best_interval(cent_value_delta, tolerance)[0]
                            ratio = str(atu.limit_format(tonal_diamond[best_idx])[0]).strip()
                            n1 = keys[tuned_pcs[inx1]]
                            n2 = keys[tuned_pcs[inx2]]
                            intervals.append((n1, n2, cent_value_delta, ratio))

                    def fmt(iv, idx):
                            n1, n2, cents, ratio = iv
                            return f"{idx:>2} {n1:>2} {n2:>2} {cents:>5} {ratio:^6}"

                    # print first and last three intervals on separate lines, nicely aligned and without Python punctuation
                    
                    print("   ".join(fmt(iv, i+1) for i, iv in enumerate(intervals[:3])))
                    print("   ".join(fmt(iv, i+1+3) for i, iv in enumerate(intervals[3:])))
        prev_chord = chord_in_cents.copy()
    return keys, root, mode

In [5]:
print(f'{numpy_dir = }')

numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/Archive/straw-man'


In [6]:
ratio_factors = np.array([ "1.25"]) # , "1.25", "1.75"
stability_factors = np.array(["0"]) # , "1.25"
max_delta = 33
snaps = np.array([0])
# suffixes = np.array(['-opt.npy']) # '-opt.npy',
suffixes = np.array(['-trans-sa-opt.npy'])
suffixes = np.array([
    'bwv257_t3_r1.375_lm19-trans-sa-opt.npy',
    'bwv258_t3_r1.250_lm19-trans-sa-opt.npy',
    'bwv256_t3_r1.500_lm19-trans-sa-opt.npy',
    'bwv263_t3_r1.125_lm19-trans-sa-opt.npy',
    'bwv264_t3_r1.500_lm17-trans-sa-opt.npy',
    'bwv254_t3_r1.625_lm17-trans-sa-opt.npy',
    'bwv259_t2_r1.250_lm19-trans-sa-opt.npy',
    'bwv255_t2_r1.375_lm19-trans-sa-opt.npy',
    'bwv260_t2_r1.500_lm17-trans-sa-opt.npy',
    'bwv261_t1_r1.125_lm17-trans-sa-opt.npy',
    'bwv253_t1_r1.125_lm17-trans-sa-opt.npy',
    'bwv262_t1_r1.750_lm19-trans-sa-opt.npy',
    ])
suffixes = np.array(['Archive/straw-man/t3_r1.125_lm19_tmp3.0/bwv258-trans-sa-opt.npy',])
# suffixes = np.array([
# 'Archive/12-TET/bwv253-12-TET-cents.npy',
# 'Archive/12-TET/bwv254-12-TET-cents.npy',
# 'Archive/12-TET/bwv255-12-TET-cents.npy',
# 'Archive/12-TET/bwv256-12-TET-cents.npy',
# 'Archive/12-TET/bwv257-12-TET-cents.npy',
# 'Archive/12-TET/bwv258-12-TET-cents.npy',
# 'Archive/12-TET/bwv259-12-TET-cents.npy',
# 'Archive/12-TET/bwv260-12-TET-cents.npy',
# 'Archive/12-TET/bwv261-12-TET-cents.npy',
# 'Archive/12-TET/bwv262-12-TET-cents.npy',
# 'Archive/12-TET/bwv263-12-TET-cents.npy',
# 'Archive/12-TET/bwv264-12-TET-cents.npy',
# ])
numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/'
limit_max = 17
tolerance = 1
measure = 0 # 0 means print all measures
print_individual_chords = True
ratios = True
print_top_notes = False
print_hits_misses = False
use_werck_top_notes = False
total_scores = 0
num_scores = 0
max_score = 0
tonal_diamond = atu.build_tonal_diamond(limit_max)
chord_scorer = atu.ChordScorer(tonal_diamond)

chord_scorer.reset_cache()
for suffix in suffixes:
    print(f'{suffix = }')
    local_numpy_dir = input_file = os.path.join(numpy_dir, f'viterbi-tunings-5-21')
    local_numpy_dir = numpy_dir
    print(f'{input_file = }, {local_numpy_dir = }')
    # Parse this file name into version, tolerance, ratio_factor, and limit_max 
    # Archive/straw-man/t3_r1.25_lm19_tmp3.0/bwv258-trans-sa-opt.npy
    version = 'bwv258'
    tolerance = 3
    ratio_factor = 1.25
    limit_max = 19
    # version = suffix.split('_')[0]
    # tolerance = int(suffix.split('_')[1][1:])
    # ratio_factor = float(suffix.split('_')[2][1:])
    # limit_max = int(suffix.split('_')[3][2:].split('-')[0])
    # limit_max = int(suffix.split('_')[3][2:].split('-')[0])

    print(f'{local_numpy_dir = }') 
    try:
        input_file = os.path.join(local_numpy_dir, f'{suffix}') 
        existing_chorale_in_cents = np.load(input_file)
        print(f'Loaded cent file from {input_file}.\nShape is {existing_chorale_in_cents.shape}') # (4,4)
        logging.info(f'{input_file = }')
    except:
        print(f'Trouble loading {input_file = }')
        continue
    num_scores += 1
    scores = np.array([chord_scorer.score_chord(chord, tolerance=tolerance) for chord in existing_chorale_in_cents.T])
    print(f'\nversion: {version}, Tol: {tolerance}, RF: {ratio_factor}, Average score: {round(np.average(scores),1)}, max score: {np.max(scores)} max chord: {np.argmax(scores)}')
    total_scores += np.average(scores)
    max_score = np.max([max_score, np.max(scores) ])
    keys, root, mode = print_chords(version, input_file, local_numpy_dir, measure, tolerance, \
            ratios=ratios, print_individual_chords=print_individual_chords, \
            use_werck_top_notes=use_werck_top_notes, print_top_notes = print_top_notes)
if print_hits_misses:
    print(f'hits and misses: {chord_scorer.return_cache_results()}')
print(f'overall total: {round(total_scores,1)}, {num_scores = }, Average Score: {round(np.average(total_scores/num_scores),1)}, {max_score = }')

suffix = 'Archive/straw-man/t3_r1.125_lm19_tmp3.0/bwv258-trans-sa-opt.npy'
input_file = '/home/prent/Repos/One-footed-bride-tuning/viterbi-tunings-5-21', local_numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/'
local_numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/'
Loaded cent file from /home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/t3_r1.125_lm19_tmp3.0/bwv258-trans-sa-opt.npy.
Shape is (4, 160)

version: bwv258, Tol: 3, RF: 1.25, Average score: 83.2, max score: 1124.0 max chord: 80

#          cents       note names   chord score
0:  208  594  208 1092	D♮ F♯ D♮ B♮	47.0
 # Fr/To Cents Ratio	 # Fr/To Cents Ratio	 # Fr/To Cents Ratio
 1 D♮ F♯   386  5/4      2 D♮ D♮     0  1/1      3 D♮ B♮   316  6/5  
 4 F♯ D♮   386  5/4      5 F♯ B♮   498  4/3      6 D♮ B♮   316  6/5  
2:  208  706  208 1092	D♮ G♮ D♮ B♮	45.0
 # Fr/To Cents Ratio	 # Fr/To Cents Ratio	 # Fr/To Cents Ratio
 1 D♮ G♮   498  4/3      2 D♮ D♮     0  1/1      3 D♮ B♮   316  6/5  
 4 G♮ D♮   498  4/3

In [7]:
# Check that cent tunings have not changed the pitch class of any note
print("Checking pitch class preservation...")
violations_found = False
for tolerance in [3]:
    for suffix in suffixes:
            try:
                input_file = os.path.join(local_numpy_dir, f'{suffix}')
                # input_file = os.path.join(local_numpy_dir, f'{version}-trans-sa-opt.npy') # -trans-sa-opt.npy
                print(f'{input_file = }')
                existing_chorale_in_cents = np.load(input_file)
            except:
                print(f'Could not load {input_file}')
                continue
            version = suffix.split('/')[2][:6]
            # version = 'bwv258'
            print(f'found {version = }')
            _, _, chorale, root, mode, keys = atu.load_chorale_in_cents(version, numpy_dir)

            violations = []
            for chord_inx, (chord_in_cents, chord_12) in enumerate(zip(existing_chorale_in_cents.T, chorale.T)):
                for voice, (cents, midi) in enumerate(zip(chord_in_cents, chord_12)):
                    original_pc = int(midi) % 12
                    tuned_pc = int(atu.pitch_class_from_cents(cents))  # half-up rounding, consistent with horizontal_transpose.py
                    if original_pc != tuned_pc:
                        violations.append((chord_inx, voice, int(midi), cents, original_pc, tuned_pc))

            if violations:
                violations_found = True
                print(f'\n{version}: {len(violations)} pitch class violation(s):')
                for chord_inx, voice, midi, cents, orig_pc, tuned_pc in violations:
                    print(f'  chord {chord_inx}, voice {voice}: MIDI {midi} ({keys[orig_pc]}) -> {cents} cents ({keys[tuned_pc]})')
            else:
                print(f'{version}: OK')

if not violations_found:
    print('\nAll pitch classes preserved across all chorales.')

Checking pitch class preservation...
input_file = '/home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/t3_r1.125_lm19_tmp3.0/bwv258-trans-sa-opt.npy'
found version = 't3_r1.'


CorpusException: Could not find a work that met this criterion: t3_r1.; if you are searching for a file on disk, use "converter" instead of "corpus".

In [23]:
import numpy as np

array_of_durations = np.array([
'05_33',
'09_25',
'07_38',
'05_36',
'11_46',
'07_24',
'03_13',
'04_45',
'03_31',
'04_03',
'06_03',
'02_38',
])

total_seconds = 0
for dur in array_of_durations:
    minutes_str, seconds_str = dur.split('_')  # Split on underscore for reliability
    total_seconds += int(minutes_str) * 60 + int(seconds_str)

# Calculate average in seconds
average_seconds = total_seconds / array_of_durations.shape[0]

# Convert average to minutes and seconds
average_minutes = int(average_seconds // 60)
remaining_seconds = int(average_seconds % 60)

# Format output with zero-padded digits (e.g., "05:03" instead of "5:3")
formatted_average = f"{average_minutes:02d}:{remaining_seconds:02d}"

print(f"Average duration: {formatted_average}")

Average duration: 05:57


In [ ]:
# Print the ratios from the root key to each of the notes in each chord.
version = 'bwv264' 
tolerance = 3
limit_max = 17
ratio_factor = '1.500'
# use: Archive/straw-man/viterbi-tunings-5-21/bwv264_t3_r1.500_lm17-trans-sa-opt.npy
numpy_dir = os.path.join('/home', 'prent','Repos', 'One-footed-bride-tuning')
cent_file = os.path.join('Archive', 'straw-man', 'viterbi-tunings-5-21', f'{version}_t{tolerance}_r{ratio_factor}_lm{limit_max}-trans-sa-opt.npy')
print(f'{(cent_file == "Archive/straw-man/viterbi-tunings-5-21/bwv264_t3_r1.500_lm17-trans-sa-opt.npy")}')
input_file = os.path.join(numpy_dir, cent_file)
# Archive/straw-man/viterbi-tunings-5-21/bwv264top-notes.npy
top_note_file = os.path.join(numpy_dir, 'Archive', 'straw-man', 'viterbi-tunings-5-21', f'{version}top-notes.npy')
# returns (chorale_in_cents, top_notes, chorale, root, mode, keys)
_, _, _, root, mode, keys = atu.load_chorale_in_cents(version, numpy_dir)
print(f'Key: {keys[root]} {mode}')
top_notes = np.load(top_note_file)
print(f'{top_notes[0] = }\n{top_notes[1] = }')
root_cent = root * 100
print(f'Root cent value: {root_cent}')
chord_in_cents = np.load(input_file).T
cent_values, cent_counts = np.unique(chord_in_cents, return_counts=True)
print(f'Cent values in chorale: {cent_values.astype(int)}')
print(f'Cent counts in chorale: {cent_counts}') 
print(f'zip of cent values and counts: {list(zip(cent_values.astype(int), cent_counts))}')
    

True
Key: G♮ major
top_notes[0] = array([ 7,  2, 11,  9,  6,  0,  4,  3,  1,  5,  8, 10])
top_notes[1] = array([ 700,  200, 1100,  900,  600,    0,  400,  300,  100,  500,  800,
       1000])
Root cent value: 700
Cent values in chorale: [   0    2    3    8   97  105  189  190  198  200  201  203  204  206
  207  208  223  224  276  364  365  372  373  379  386  388  389  394
  576  584  590  592  593  595  610  618  680  688  689  695  696  698
  699  701  702  704  705  706  710  714  722  884  892  900  906  908
  909  926  947 1066 1074 1075 1082 1084 1085 1087 1088 1090 1091 1092
 1108 1158 1172 1175 1192 1193]
Cent counts in chorale: [ 8  2  8  8  1  1  1 14 32  4 12 26 16 21 14  8  1 12  4  3  2  2  2  4
  7  6  4  4  2 12  7 10  2  1  4  2  8  5  4  2 16  8 24 36 16 12 10 16
  2  1  8  4  8 12  8  8  4  8  2  2  4  2  8  4 12 20  8 18  4  8  4  1
  2  1  1  4]
zip of cent values and counts: [(0, 8), (2, 2), (3, 8), (8, 8), (97, 1), (105, 1), (189, 1), (190, 14), (198, 32), (200